In [0]:
%run ./00_config

✅ Config loaded : billing.sales_data
📅 Target date   : 2026-10-03


In [0]:
# ============================================================
# GOLD CATEGORY PERFORMANCE
# Production-Grade Databricks Serverless Pipeline
# ============================================================
#
# SOURCE:
#   billing.sales_data.billing_silver
#
# TARGET:
#   billing.sales_data.gold_category_performance
#
# TARGET SCHEMA:
#   subCategoryOf : STRING
#   orderDate     : DATE
#   sales         : DOUBLE
#   source_date   : STRING
#
# BUSINESS LOGIC:
#   - No orderStatus filter
#   - Exclude:
#       1. ho marlboro
#       2. dummy store --- for testing only
#   - Sales = SUM(totalProductPrice)
#   - NULL / blank subCategoryOf -> "Unknown"
#   - source_date = TARGET_DATE
#
# WRITE STRATEGY:
#   - Atomic Delta replaceWhere
#   - No DELETE + INSERT
#   - No mergeSchema
#   - No overwriteSchema
#   - No cache()
#   - No persist()
#   - No unpersist()
#
# VALIDATIONS:
#   - Source table availability
#   - Required source columns
#   - Target date availability
#   - Excluded stores
#   - NULL orderDate
#   - NULL sales
#   - NULL/blank category normalization
#   - Gold NULL grain
#   - Duplicate grain
#   - Silver -> Gold sales reconciliation
#   - Existing target schema
#   - Post-write row count
#   - Post-write sales
#   - Post-write schema
#   - Post-write NULL grain
#   - Post-write duplicate grain
#   - Post-write source_date
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid
import time


# ============================================================
# 2. CONFIGURATION
# ============================================================

SOURCE_TABLE = "billing.sales_data.billing_silver"

TARGET_TABLE = "billing.sales_data.gold_category_performance"

TARGET_DATE = "2026-10-03"

EXCLUDED_STORES = [
    "ho marlboro",
    "dummy store --- for testing only"
]

RECONCILIATION_TOLERANCE = 0.01

PIPELINE_RUN_ID = str(uuid.uuid4())

START_TIME = time.time()


# ============================================================
# 3. LOGGING
# ============================================================

def log(message):
    now = datetime.now(timezone.utc).strftime(
        "%Y-%m-%d %H:%M:%S UTC"
    )
    print(
        f"[GOLD_CATEGORY] [{now}] {message}"
    )


def fail(message):
    raise RuntimeError(
        f"[GOLD_CATEGORY] {message}"
    )


# ============================================================
# 4. START
# ============================================================

log("============================================================")
log("Starting Gold Category Performance Pipeline")
log("============================================================")

log(f"Pipeline Run ID : {PIPELINE_RUN_ID}")
log(f"Target Date     : {TARGET_DATE}")
log(f"Source Table    : {SOURCE_TABLE}")
log(f"Target Table    : {TARGET_TABLE}")

log("============================================================")


# ============================================================
# 5. VALIDATE TARGET DATE
# ============================================================

try:

    target_date = datetime.strptime(
        TARGET_DATE,
        "%Y-%m-%d"
    ).date()

except Exception:

    fail(
        f"Invalid TARGET_DATE='{TARGET_DATE}'. "
        f"Expected YYYY-MM-DD."
    )


log(
    f"Validated TARGET_DATE = {target_date}"
)


# ============================================================
# 6. READ SILVER
# ============================================================

log("Reading Silver source table...")

try:

    silver_df = spark.table(
        SOURCE_TABLE
    )

except Exception as e:

    fail(
        f"Unable to read source table: {e}"
    )


silver_count = silver_df.count()

log(
    f"Silver source rows = {silver_count:,}"
)


if silver_count == 0:

    fail(
        "Silver source table contains zero rows."
    )


# ============================================================
# 7. REQUIRED COLUMN VALIDATION
# ============================================================

required_columns = [
    "orderDate",
    "subCategoryOf",
    "totalProductPrice",
    "storeName"
]


source_columns = set(
    silver_df.columns
)


missing_columns = [
    column
    for column in required_columns
    if column not in source_columns
]


if missing_columns:

    fail(
        "Missing required Silver columns: "
        f"{missing_columns}"
    )


log(
    "Required Silver columns validated."
)


# ============================================================
# 8. NORMALIZE ORDER DATE
# ============================================================

silver_prepared = (
    silver_df
    .withColumn(
        "_order_date",
        F.to_date(
            F.col("orderDate")
        )
    )
)


# ============================================================
# 9. FILTER TARGET DATE
# ============================================================

target_date_df = (
    silver_prepared
    .filter(
        F.col("_order_date")
        == F.lit(TARGET_DATE).cast("date")
    )
)


target_date_count = target_date_df.count()


log(
    f"Silver rows for {TARGET_DATE} = "
    f"{target_date_count:,}"
)


if target_date_count == 0:

    fail(
        f"No Silver records found for "
        f"TARGET_DATE={TARGET_DATE}"
    )


# ============================================================
# 10. NORMALIZE STORE NAME
# ============================================================

filtered = (
    target_date_df
    .withColumn(
        "_store_normalized",
        F.lower(
            F.trim(
                F.coalesce(
                    F.col("storeName")
                    .cast("string"),
                    F.lit("")
                )
            )
        )
    )
)


# ============================================================
# 11. EXCLUDED STORE DIAGNOSTIC
# ============================================================

excluded_store_count = (
    filtered
    .filter(
        F.col("_store_normalized")
        .isin(EXCLUDED_STORES)
    )
    .count()
)


log(
    f"Rows belonging to excluded stores = "
    f"{excluded_store_count:,}"
)


# ============================================================
# 12. REMOVE EXCLUDED STORES
# ============================================================

filtered = (
    filtered
    .filter(
        ~F.col("_store_normalized")
        .isin(EXCLUDED_STORES)
    )
)


filtered_count = filtered.count()


log(
    f"Rows after excluded-store filtering = "
    f"{filtered_count:,}"
)


if filtered_count == 0:

    fail(
        "No rows remain after excluded-store filtering."
    )


# ============================================================
# 13. EXCLUDED STORE VALIDATION
# ============================================================

remaining_excluded_rows = (
    filtered
    .filter(
        F.col("_store_normalized")
        .isin(EXCLUDED_STORES)
    )
    .count()
)


if remaining_excluded_rows > 0:

    fail(
        f"Excluded-store validation failed. "
        f"{remaining_excluded_rows:,} excluded-store "
        f"rows remain."
    )


log(
    "Excluded-store validation passed."
)


# ============================================================
# 14. NULL ORDER DATE VALIDATION
# ============================================================

null_order_date_count = (
    filtered
    .filter(
        F.col("_order_date").isNull()
    )
    .count()
)


if null_order_date_count > 0:

    fail(
        f"Found {null_order_date_count:,} rows with "
        f"NULL orderDate."
    )


log(
    "orderDate NULL validation passed."
)


# ============================================================
# 15. NULL SALES VALIDATION
# ============================================================

null_sales_count = (
    filtered
    .filter(
        F.col("totalProductPrice").isNull()
    )
    .count()
)


if null_sales_count > 0:

    fail(
        f"Found {null_sales_count:,} rows with "
        f"NULL totalProductPrice."
    )


log(
    "totalProductPrice NULL validation passed."
)


# ============================================================
# 16. NULL / BLANK SUBCATEGORY ANALYSIS
# ============================================================
#
# NULL/blank subCategoryOf is NOT treated as a fatal error.
#
# Instead:
#
#     NULL       -> Unknown
#     ""         -> Unknown
#     "   "      -> Unknown
#
# This prevents sales from being lost.
# ============================================================

null_category_filter = (
    F.col("subCategoryOf").isNull()
    |
    (
        F.trim(
            F.col("subCategoryOf")
            .cast("string")
        ) == ""
    )
)


null_category_count = (
    filtered
    .filter(
        null_category_filter
    )
    .count()
)


log(
    f"Rows with NULL/blank subCategoryOf = "
    f"{null_category_count:,}"
)


# ============================================================
# 17. NULL CATEGORY IMPACT ANALYSIS
# ============================================================

null_category_stats = (
    filtered
    .filter(
        null_category_filter
    )
    .agg(
        F.count("*").alias(
            "rows"
        ),

        F.sum(
            F.col("totalProductPrice")
            .cast("double")
        ).alias(
            "sales"
        ),

        F.countDistinct(
            "storeName"
        ).alias(
            "stores"
        )
    )
    .first()
)


if null_category_stats:

    impact_rows = (
        null_category_stats["rows"]
        or 0
    )

    impact_sales = (
        float(
            null_category_stats["sales"]
        )
        if null_category_stats["sales"] is not None
        else 0.0
    )

    impact_stores = (
        null_category_stats["stores"]
        or 0
    )

    log(
        "NULL/blank category impact:"
    )

    log(
        f"  Rows   = {impact_rows:,}"
    )

    log(
        f"  Sales  = {impact_sales:,.4f}"
    )

    log(
        f"  Stores = {impact_stores:,}"
    )


# ============================================================
# 18. PREPARE BUSINESS DATA
# ============================================================

prepared = (
    filtered

    # --------------------------------------------------------
    # Order date
    # --------------------------------------------------------

    .withColumn(
        "_order_date",
        F.to_date(
            F.col("orderDate")
        )
    )

    # --------------------------------------------------------
    # Sales
    # --------------------------------------------------------

    .withColumn(
        "_sales",
        F.col("totalProductPrice")
        .cast("double")
    )

    # --------------------------------------------------------
    # Subcategory
    #
    # NULL / blank -> Unknown
    # --------------------------------------------------------

    .withColumn(
        "_subcategory",
        F.when(
            F.col("subCategoryOf").isNull()
            |
            (
                F.trim(
                    F.col("subCategoryOf")
                    .cast("string")
                ) == ""
            ),
            F.lit("Unknown")
        )
        .otherwise(
            F.trim(
                F.col("subCategoryOf")
                .cast("string")
            )
        )
        .cast("string")
    )
)


# ============================================================
# 19. PREPARED DATA VALIDATION
# ============================================================

prepared_null_order_date = (
    prepared
    .filter(
        F.col("_order_date").isNull()
    )
    .count()
)


if prepared_null_order_date > 0:

    fail(
        f"Prepared data contains "
        f"{prepared_null_order_date:,} "
        f"NULL orderDate values."
    )


prepared_null_sales = (
    prepared
    .filter(
        F.col("_sales").isNull()
    )
    .count()
)


if prepared_null_sales > 0:

    fail(
        f"Prepared data contains "
        f"{prepared_null_sales:,} "
        f"NULL sales values."
    )


prepared_null_category = (
    prepared
    .filter(
        F.col("_subcategory").isNull()
    )
    .count()
)


if prepared_null_category > 0:

    fail(
        f"Prepared data contains "
        f"{prepared_null_category:,} "
        f"NULL subcategory values."
    )


log(
    "Prepared-data validation passed."
)


# ============================================================
# 20. SILVER SALES BASELINE
# ============================================================

silver_sales_row = (
    prepared
    .agg(
        F.sum(
            "_sales"
        ).alias(
            "sales"
        )
    )
    .first()
)


silver_sales = (
    float(
        silver_sales_row["sales"]
    )
    if silver_sales_row["sales"] is not None
    else 0.0
)


log(
    f"Silver filtered sales = "
    f"{silver_sales:,.4f}"
)


# ============================================================
# 21. AGGREGATE TO GOLD
# ============================================================
#
# GOLD GRAIN:
#
#   subCategoryOf + orderDate
#
# ============================================================

gold_df = (
    prepared

    .groupBy(
        "_subcategory",
        "_order_date"
    )

    .agg(
        F.sum(
            "_sales"
        )
        .cast("double")
        .alias(
            "sales"
        )
    )

    .select(
        F.col("_subcategory")
        .cast("string")
        .alias(
            "subCategoryOf"
        ),

        F.col("_order_date")
        .cast("date")
        .alias(
            "orderDate"
        ),

        F.col("sales")
        .cast("double")
        .alias(
            "sales"
        ),

        F.lit(
            TARGET_DATE
        )
        .cast("string")
        .alias(
            "source_date"
        )
    )
)


# ============================================================
# 22. GOLD ROW COUNT
# ============================================================

gold_count = gold_df.count()


log(
    f"Generated Gold rows = "
    f"{gold_count:,}"
)


if gold_count == 0:

    fail(
        "Generated Gold DataFrame contains zero rows."
    )


# ============================================================
# 23. EXPECTED GOLD SCHEMA
# ============================================================

expected_columns = [
    "subCategoryOf",
    "orderDate",
    "sales",
    "source_date"
]


expected_schema = {
    "subCategoryOf": "string",
    "orderDate": "date",
    "sales": "double",
    "source_date": "string"
}


# ============================================================
# 24. GENERATED GOLD COLUMN ORDER
# ============================================================

actual_columns = gold_df.columns


if actual_columns != expected_columns:

    fail(
        "Generated Gold column order mismatch.\n"
        f"Expected: {expected_columns}\n"
        f"Actual:   {actual_columns}"
    )


# ============================================================
# 25. GENERATED GOLD DATA TYPES
# ============================================================

actual_schema = {
    field.name: field.dataType.simpleString()
    for field in gold_df.schema.fields
}


schema_mismatches = []


for column_name, expected_type in expected_schema.items():

    actual_type = (
        actual_schema.get(
            column_name
        )
    )

    if actual_type != expected_type:

        schema_mismatches.append(
            (
                column_name,
                expected_type,
                actual_type
            )
        )


if schema_mismatches:

    mismatch_text = "\n".join(
        [
            (
                f"{column}: "
                f"expected={expected}, "
                f"actual={actual}"
            )
            for column, expected, actual
            in schema_mismatches
        ]
    )

    fail(
        "Generated Gold schema mismatch:\n"
        + mismatch_text
    )


log(
    "Generated Gold schema validated."
)


# ============================================================
# 26. GOLD NULL GRAIN VALIDATION
# ============================================================

null_grain_rows = (
    gold_df
    .filter(
        F.col("subCategoryOf").isNull()
        |
        F.col("orderDate").isNull()
        |
        F.col("source_date").isNull()
    )
    .count()
)


if null_grain_rows > 0:

    fail(
        f"Gold contains "
        f"{null_grain_rows:,} rows with "
        f"NULL grain keys."
    )


log(
    "Gold NULL-grain validation passed."
)


# ============================================================
# 27. GOLD SOURCE DATE VALIDATION
# ============================================================

wrong_source_date_rows = (
    gold_df
    .filter(
        F.col("source_date")
        != F.lit(TARGET_DATE)
    )
    .count()
)


if wrong_source_date_rows > 0:

    fail(
        f"Gold contains "
        f"{wrong_source_date_rows:,} rows with "
        f"incorrect source_date."
    )


log(
    "Gold source_date validation passed."
)


# ============================================================
# 28. GOLD DUPLICATE GRAIN VALIDATION
# ============================================================

duplicate_grain_groups = (
    gold_df
    .groupBy(
        "subCategoryOf",
        "orderDate"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


if duplicate_grain_groups > 0:

    fail(
        f"Gold contains "
        f"{duplicate_grain_groups:,} duplicate "
        f"(subCategoryOf, orderDate) grain groups."
    )


log(
    "Gold duplicate-grain validation passed."
)


# ============================================================
# 29. GOLD SALES
# ============================================================

gold_sales_row = (
    gold_df
    .agg(
        F.sum(
            "sales"
        ).alias(
            "sales"
        )
    )
    .first()
)


gold_sales = (
    float(
        gold_sales_row["sales"]
    )
    if gold_sales_row["sales"] is not None
    else 0.0
)


log(
    f"Generated Gold sales = "
    f"{gold_sales:,.4f}"
)


# ============================================================
# 30. SILVER -> GOLD RECONCILIATION
# ============================================================

sales_difference = abs(
    silver_sales - gold_sales
)


log(
    f"Silver sales     = "
    f"{silver_sales:,.4f}"
)

log(
    f"Gold sales       = "
    f"{gold_sales:,.4f}"
)

log(
    f"Sales difference = "
    f"{sales_difference:,.4f}"
)


if sales_difference > RECONCILIATION_TOLERANCE:

    fail(
        "Silver -> Gold sales reconciliation FAILED.\n"
        f"Silver = {silver_sales:,.4f}\n"
        f"Gold   = {gold_sales:,.4f}\n"
        f"Diff   = {sales_difference:,.4f}\n"
        f"Tolerance = {RECONCILIATION_TOLERANCE}"
    )


log(
    "Silver -> Gold sales reconciliation PASSED."
)


# ============================================================
# 31. CHECK TARGET TABLE
# ============================================================

target_exists = (
    spark.catalog.tableExists(
        TARGET_TABLE
    )
)


log(
    f"Target table exists = "
    f"{target_exists}"
)


# ============================================================
# 32. VALIDATE EXISTING TARGET SCHEMA
# ============================================================

if target_exists:

    log(
        "Reading existing target schema..."
    )

    target_df = spark.table(
        TARGET_TABLE
    )

    target_columns = (
        target_df.columns
    )

    if target_columns != expected_columns:

        fail(
            "Existing target column order mismatch.\n"
            f"Expected: {expected_columns}\n"
            f"Actual:   {target_columns}"
        )


    target_schema = {
        field.name: field.dataType.simpleString()
        for field in target_df.schema.fields
    }


    target_schema_mismatches = []


    for column_name, expected_type in expected_schema.items():

        actual_type = (
            target_schema.get(
                column_name
            )
        )

        if actual_type != expected_type:

            target_schema_mismatches.append(
                (
                    column_name,
                    expected_type,
                    actual_type
                )
            )


    if target_schema_mismatches:

        mismatch_text = "\n".join(
            [
                (
                    f"{column}: "
                    f"expected={expected}, "
                    f"actual={actual}"
                )
                for column, expected, actual
                in target_schema_mismatches
            ]
        )

        fail(
            "Existing target schema mismatch:\n"
            + mismatch_text
        )


    log(
        "Existing target schema matches expected production schema."
    )

else:

    log(
        "Target table does not exist."
    )

    log(
        "It will be created using the generated schema."
    )


# ============================================================
# 33. PRE-WRITE TARGET METRICS
# ============================================================

if target_exists:

    pre_write_target_df = (
        spark.table(
            TARGET_TABLE
        )
        .filter(
            F.col("source_date")
            == F.lit(TARGET_DATE)
        )
    )


    pre_write_count = (
        pre_write_target_df.count()
    )


    pre_write_sales_row = (
        pre_write_target_df
        .agg(
            F.sum(
                "sales"
            ).alias(
                "sales"
            )
        )
        .first()
    )


    pre_write_sales = (
        float(
            pre_write_sales_row["sales"]
        )
        if pre_write_sales_row["sales"] is not None
        else 0.0
    )

else:

    pre_write_count = 0

    pre_write_sales = 0.0


log(
    f"Existing target rows for "
    f"{TARGET_DATE} = "
    f"{pre_write_count:,}"
)

log(
    f"Existing target sales for "
    f"{TARGET_DATE} = "
    f"{pre_write_sales:,.4f}"
)


# ============================================================
# 34. ATOMIC DELTA REPLACEMENT
# ============================================================
#
# IMPORTANT:
#
#   mode("overwrite")
#   replaceWhere = source_date = TARGET_DATE
#
# This replaces ONLY the target date.
#
# We intentionally do NOT use:
#
#   mergeSchema=True
#   overwriteSchema=True
#
# ============================================================

log(
    "Starting atomic Delta replacement..."
)

log(
    f"replaceWhere: "
    f"source_date = '{TARGET_DATE}'"
)


write_start = time.time()


try:

    (
        gold_df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"source_date = '{TARGET_DATE}'"
        )
        .saveAsTable(
            TARGET_TABLE
        )
    )


except Exception as e:

    fail(
        "Atomic Delta replacement FAILED:\n"
        f"{e}"
    )


write_duration = (
    time.time() - write_start
)


log(
    "Atomic Delta replacement SUCCESS."
)

log(
    f"Write duration = "
    f"{write_duration:.2f} seconds"
)


# ============================================================
# 35. POST-WRITE READ
# ============================================================

log(
    "Reading target table for post-write validation..."
)


post_write_df = (
    spark.table(
        TARGET_TABLE
    )
    .filter(
        F.col("source_date")
        == F.lit(TARGET_DATE)
    )
)


# ============================================================
# 36. POST-WRITE ROW COUNT
# ============================================================

post_write_count = (
    post_write_df.count()
)


log(
    f"Expected Gold rows = "
    f"{gold_count:,}"
)

log(
    f"Actual Gold rows   = "
    f"{post_write_count:,}"
)


if post_write_count != gold_count:

    fail(
        "Post-write row-count validation FAILED.\n"
        f"Expected = {gold_count:,}\n"
        f"Actual   = {post_write_count:,}"
    )


log(
    "Post-write row-count validation PASSED."
)


# ============================================================
# 37. POST-WRITE SALES
# ============================================================

post_write_sales_row = (
    post_write_df
    .agg(
        F.sum(
            "sales"
        ).alias(
            "sales"
        )
    )
    .first()
)


post_write_sales = (
    float(
        post_write_sales_row["sales"]
    )
    if post_write_sales_row["sales"] is not None
    else 0.0
)


post_write_sales_difference = abs(
    gold_sales - post_write_sales
)


log(
    f"Generated Gold sales = "
    f"{gold_sales:,.4f}"
)

log(
    f"Post-write Gold sales = "
    f"{post_write_sales:,.4f}"
)

log(
    f"Post-write difference = "
    f"{post_write_sales_difference:,.4f}"
)


if (
    post_write_sales_difference
    > RECONCILIATION_TOLERANCE
):

    fail(
        "Post-write sales validation FAILED.\n"
        f"Expected = {gold_sales:,.4f}\n"
        f"Actual   = {post_write_sales:,.4f}\n"
        f"Difference = "
        f"{post_write_sales_difference:,.4f}"
    )


log(
    "Post-write sales validation PASSED."
)


# ============================================================
# 38. POST-WRITE SCHEMA VALIDATION
# ============================================================

final_target_df = (
    spark.table(
        TARGET_TABLE
    )
)


final_columns = (
    final_target_df.columns
)


if final_columns != expected_columns:

    fail(
        "Post-write target column order mismatch.\n"
        f"Expected: {expected_columns}\n"
        f"Actual:   {final_columns}"
    )


final_schema = {
    field.name: field.dataType.simpleString()
    for field in final_target_df.schema.fields
}


final_schema_mismatches = []


for column_name, expected_type in expected_schema.items():

    actual_type = (
        final_schema.get(
            column_name
        )
    )

    if actual_type != expected_type:

        final_schema_mismatches.append(
            (
                column_name,
                expected_type,
                actual_type
            )
        )


if final_schema_mismatches:

    mismatch_text = "\n".join(
        [
            (
                f"{column}: "
                f"expected={expected}, "
                f"actual={actual}"
            )
            for column, expected, actual
            in final_schema_mismatches
        ]
    )

    fail(
        "Post-write target schema mismatch:\n"
        + mismatch_text
    )


log(
    "Post-write schema validation PASSED."
)


# ============================================================
# 39. POST-WRITE NULL GRAIN VALIDATION
# ============================================================

post_write_null_grain = (
    post_write_df
    .filter(
        F.col("subCategoryOf").isNull()
        |
        F.col("orderDate").isNull()
        |
        F.col("source_date").isNull()
    )
    .count()
)


if post_write_null_grain > 0:

    fail(
        f"Post-write validation found "
        f"{post_write_null_grain:,} "
        f"rows with NULL grain keys."
    )


log(
    "Post-write NULL-grain validation PASSED."
)


# ============================================================
# 40. POST-WRITE DUPLICATE GRAIN VALIDATION
# ============================================================

post_write_duplicate_groups = (
    post_write_df
    .groupBy(
        "subCategoryOf",
        "orderDate"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


if post_write_duplicate_groups > 0:

    fail(
        f"Post-write validation found "
        f"{post_write_duplicate_groups:,} "
        f"duplicate grain groups."
    )


log(
    "Post-write duplicate-grain validation PASSED."
)


# ============================================================
# 41. POST-WRITE SOURCE DATE VALIDATION
# ============================================================

post_write_wrong_source_date = (
    post_write_df
    .filter(
        F.col("source_date")
        != F.lit(TARGET_DATE)
    )
    .count()
)


if post_write_wrong_source_date > 0:

    fail(
        f"Post-write validation found "
        f"{post_write_wrong_source_date:,} "
        f"rows with incorrect source_date."
    )


log(
    "Post-write source_date validation PASSED."
)


# ============================================================
# 42. UNKNOWN CATEGORY METRICS
# ============================================================

unknown_metrics = (
    post_write_df
    .filter(
        F.col("subCategoryOf")
        == "Unknown"
    )
    .agg(
        F.count("*").alias(
            "rows"
        ),

        F.sum(
            "sales"
        ).alias(
            "sales"
        )
    )
    .first()
)


unknown_rows = (
    unknown_metrics["rows"]
    or 0
)


unknown_sales = (
    float(
        unknown_metrics["sales"]
    )
    if unknown_metrics["sales"] is not None
    else 0.0
)


log(
    f"Unknown category Gold rows = "
    f"{unknown_rows:,}"
)

log(
    f"Unknown category Gold sales = "
    f"{unknown_sales:,.4f}"
)


# ============================================================
# 43. CATEGORY SUMMARY
# ============================================================

log(
    "Generating category summary..."
)


category_summary = (
    post_write_df
    .groupBy(
        "subCategoryOf"
    )
    .agg(
        F.sum(
            "sales"
        ).alias(
            "sales"
        ),

        F.count("*").alias(
            "rows"
        )
    )
    .orderBy(
        F.col("sales").desc()
    )
)


display(
    category_summary
)


# ============================================================
# 44. FINAL METRICS
# ============================================================

duration = (
    time.time() - START_TIME
)


log("============================================================")
log("GOLD CATEGORY PERFORMANCE PIPELINE COMPLETED")
log("============================================================")

log(
    f"Pipeline Run ID           : "
    f"{PIPELINE_RUN_ID}"
)

log(
    f"Target Date               : "
    f"{TARGET_DATE}"
)

log(
    f"Silver source rows        : "
    f"{silver_count:,}"
)

log(
    f"Target-date Silver rows   : "
    f"{target_date_count:,}"
)

log(
    f"Rows after exclusions     : "
    f"{filtered_count:,}"
)

log(
    f"Generated Gold rows       : "
    f"{gold_count:,}"
)

log(
    f"Post-write Gold rows      : "
    f"{post_write_count:,}"
)

log(
    f"Silver filtered sales     : "
    f"{silver_sales:,.4f}"
)

log(
    f"Generated Gold sales      : "
    f"{gold_sales:,.4f}"
)

log(
    f"Post-write Gold sales     : "
    f"{post_write_sales:,.4f}"
)

log(
    f"Sales difference           : "
    f"{sales_difference:,.4f}"
)

log(
    f"Unknown category rows     : "
    f"{unknown_rows:,}"
)

log(
    f"Unknown category sales    : "
    f"{unknown_sales:,.4f}"
)

log(
    f"Write duration            : "
    f"{write_duration:.2f} seconds"
)

log(
    f"Total pipeline duration   : "
    f"{duration:.2f} seconds"
)

log(
    "Atomic replacement        : SUCCESS"
)

log(
    "Schema validation         : SUCCESS"
)

log(
    "Sales reconciliation      : SUCCESS"
)

log(
    "Duplicate grain check     : SUCCESS"
)

log(
    "Post-write validation     : SUCCESS"
)

log("============================================================")

[GOLD_CATEGORY] [2026-10-04 11:00:42 UTC] ============================================================
[GOLD_CATEGORY] [2026-10-04 11:00:42 UTC] Starting Gold Category Performance Pipeline
[GOLD_CATEGORY] [2026-10-04 11:00:42 UTC] ============================================================
[GOLD_CATEGORY] [2026-10-04 11:00:42 UTC] Pipeline Run ID : 34cc93c6-850a-4b68-a67c-c4e2aadd321f
[GOLD_CATEGORY] [2026-10-04 11:00:42 UTC] Target Date     : 2026-10-03
[GOLD_CATEGORY] [2026-10-04 11:00:42 UTC] Source Table    : billing.sales_data.billing_silver
[GOLD_CATEGORY] [2026-10-04 11:00:42 UTC] Target Table    : billing.sales_data.gold_category_performance
[GOLD_CATEGORY] [2026-10-04 11:00:42 UTC] ============================================================
[GOLD_CATEGORY] [2026-10-04 11:00:42 UTC] Validated TARGET_DATE = 2026-10-03
[GOLD_CATEGORY] [2026-10-04 11:00:42 UTC] Reading Silver source table...
[GOLD_CATEGORY] [2026-10-04 11:00:43 UTC] Silver source rows = 25,098,886
[GOLD_CATEGORY

subCategoryOf,sales,rows
Beverages,1242065.66,1
Paan,807067.14,1
Snacks,543130.5799999998,1
"Sweets, Chocolates & Candies",495732.6599999999,1
Ice Creams & Desserts,218489.5100000001,1
Cafe,213020.22,1
Breakfast & Dairy,182084.68000000002,1
Bakery & Gourmet,110155.64,1
Imported,90662.78,1
Instant Foods & Ready To Eat,85776.08999999998,1


[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] ============================================================
[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] GOLD CATEGORY PERFORMANCE PIPELINE COMPLETED
[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] ============================================================
[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] Pipeline Run ID           : 34cc93c6-850a-4b68-a67c-c4e2aadd321f
[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] Target Date               : 2026-10-03
[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] Silver source rows        : 25,098,886
[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] Target-date Silver rows   : 52,995
[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] Rows after exclusions     : 52,995
[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] Generated Gold rows       : 43
[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] Post-write Gold rows      : 43
[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] Silver filtered sales     : 4,496,998.8700
[GOLD_CATEGORY] [2026-10-04 11:01:02 UTC] Generated G